## NC Intervention Experiment

Tests whether the feature norm fn* **causally** controls neural collapse, or is merely a correlated progress measure.

**Protocol:** Train MLP-5 through Phase 1 (CE, 200 epochs), save checkpoint, then run 3 Phase-2 conditions:
- `control`: normal MSE training
- `scale_down` (α=0.3): last hidden layer weights × 0.3 → fn pushed down
- `scale_up` (α=3.0): last hidden layer weights × 3.0 → fn pushed up

**Causal prediction:** Scale DOWN → earlier T_NC; Scale UP → later T_NC

**Correlation prediction:** All conditions collapse at similar T_NC (fn* is equilibrium property)

Run on Colab T4 or Colab GPU (~45 min).

In [1]:
"""
NC Intervention Experiment
==========================
Tests whether fn causally controls collapse, or is merely correlated.

Protocol:
  Phase 1: CE training (200 epochs) — shared across all conditions
  Phase 2: MSE training — 3 conditions from the same checkpoint:
    (a) Control   — no intervention
    (b) Scale DOWN (alpha=0.3) — last hidden layer weights * 0.3
                                  pushes fn from ~12 to ~3.6 (below fn*)
    (c) Scale UP   (alpha=3.0) — last hidden layer weights * 3.0
                                  pushes fn from ~12 to ~36 (far above fn*)

  Each condition run with 3 seeds (3 x 3 = 9 Phase-2 runs total).

Causal prediction:
  If fn drives collapse: Scale DOWN → earlier T_NC, Scale UP → later T_NC
  If fn is a correlate:  All three conditions collapse at same T_NC

Run on Colab T4 or Colab GPU. Takes ~45 min.
"""

import torch, torchvision, time, copy
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/content/'   # change to '/content/' for Colab
print(f'Device: {DEVICE}')


Device: cuda


In [2]:
# ── Data ──────────────────────────────────────────────────────────────────
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/content/data', train=True,
                                        download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/content/data', train=False,
                                        download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f'MNIST: {len(trainset):,} train / {len(testset):,} test')


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.63MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 130kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.82MB/s]

MNIST: 60,000 train / 10,000 test


In [3]:
# ── Model ──────────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, depth=5, width=512):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.ReLU()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, 10)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()


In [4]:
# ── NC metrics ─────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)))
        ll.append(y.to(DEVICE, non_blocking=True))
    H = torch.cat(fl); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool, device=DEVICE)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().to(DEVICE), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3,
            'feat_norm': H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item(); total += len(y)
    return correct / total


In [5]:
# ── Phase 1: shared CE training (200 epochs, 3 seeds) ────────────────────
print('\n' + '='*55)
print('PHASE 1: CE training (shared across all conditions)')
print('='*55)

phase1_checkpoints = {}   # seed -> state_dict
phase1_fn = {}            # seed -> fn at end of Phase 1

for seed in range(3):
    print(f'\n--- Phase 1, seed={seed} ---')
    torch.manual_seed(seed)
    model = MLP().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)
    t0    = time.time()

    for ep in range(1, 201):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(x), y).backward()
            opt.step()
        sch.step()

    tr = evaluate(model, train_loader)
    nc = compute_nc(model, train_loader)
    print(f'  => train={tr:.4f}  fn={nc["feat_norm"]:.3f}  NC1={nc["nc1"]:.4f}'
          f'  ({time.time()-t0:.0f}s)')

    phase1_checkpoints[seed] = copy.deepcopy(model.state_dict())
    phase1_fn[seed] = nc['feat_norm']

print(f'\nPhase 1 fn values: {[f"{v:.3f}" for v in phase1_fn.values()]}')
print(f'Mean fn at Phase 1 end: {np.mean(list(phase1_fn.values())):.3f}')



PHASE 1: CE training (shared across all conditions)

--- Phase 1, seed=0 ---
  => train=1.0000  fn=16.189  NC1=0.1067  (639s)

--- Phase 1, seed=1 ---
  => train=1.0000  fn=16.627  NC1=0.1186  (642s)

--- Phase 1, seed=2 ---
  => train=1.0000  fn=15.921  NC1=0.1053  (642s)

Phase 1 fn values: ['16.189', '16.627', '15.921']
Mean fn at Phase 1 end: 16.245


In [6]:
# ── Phase 2: intervention experiment ──────────────────────────────────────
print('\n' + '='*55)
print('PHASE 2: Intervention experiment')
print('='*55)

# Scale factors: control (1.0), scale down (0.3), scale up (3.0)
# alpha scales the last Linear layer in body (index -1 or the last nn.Linear)
# This directly multiplies fn at Phase 2 start by alpha.
ALPHAS = {
    'control':    1.0,
    'scale_down': 0.3,   # fn: ~12 * 0.3 = ~3.6  (below fn*=1.06 after further decay)
    'scale_up':   3.0,   # fn: ~12 * 3.0 = ~36   (far above fn*)
}
NC1_THRESH = 0.01
PHASE2_EPOCHS = 400

all_results = []

for condition, alpha in ALPHAS.items():
    for seed in range(3):
        print(f'\n--- condition={condition} (alpha={alpha})  seed={seed} ---')

        # Load Phase 1 checkpoint
        model = MLP().to(DEVICE)
        model.load_state_dict(phase1_checkpoints[seed])

        # ── Apply intervention: rescale last hidden layer weights ──────────
        # Find the last Linear layer in body (before head)
        last_linear = None
        for m in model.body.modules():
            if isinstance(m, nn.Linear):
                last_linear = m
        assert last_linear is not None

        with torch.no_grad():
            last_linear.weight.data *= alpha
            last_linear.bias.data   *= alpha

        # Verify fn changed as expected
        nc_post = compute_nc(model, train_loader)
        fn_before = phase1_fn[seed]
        fn_after  = nc_post['feat_norm']
        print(f'  fn before rescaling: {fn_before:.3f}')
        print(f'  fn after  rescaling: {fn_after:.3f}  (ratio: {fn_after/fn_before:.2f}x)')

        # ── Phase 2: MSE training ──────────────────────────────────────────
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE2_EPOCHS)

        rows = []; t_nc = None; t0 = time.time()
        for ep_local in range(1, PHASE2_EPOCHS + 1):
            ep = 200 + ep_local
            model.train()
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                F.mse_loss(logits, F.one_hot(y, 10).float()).backward()
                opt.step()
            sch.step()

            if ep_local % 10 == 0 or ep_local == PHASE2_EPOCHS:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                nc = compute_nc(model, train_loader)
                rows.append({'epoch': ep, 'condition': condition, 'alpha': alpha,
                             'seed': seed, 'train': tr, 'test': te, **nc})
                if t_nc is None and nc['nc1'] < NC1_THRESH:
                    t_nc = ep
                    print(f'  NC1 collapsed at epoch {ep}  fn={nc["feat_norm"]:.4f}')

        fn_at_tnc = next((r['feat_norm'] for r in rows
                          if r['epoch'] == t_nc), None) if t_nc else None
        status = f'T_NC={t_nc}  fn*={fn_at_tnc:.4f}' if t_nc else 'DNF'
        print(f'  => {status}  ({time.time()-t0:.0f}s)')

        df = pd.DataFrame(rows)
        fname = f'intervention_{condition}_s{seed}.csv'
        df.to_csv(SAVE_DIR + fname, index=False)

        all_results.append({
            'condition': condition, 'alpha': alpha, 'seed': seed,
            'fn_before': fn_before, 'fn_after_rescale': fn_after,
            'T_NC': t_nc, 'fn_at_TNC': fn_at_tnc,
            'test_acc': rows[-1]['test'] if rows else None
        })



PHASE 2: Intervention experiment

--- condition=control (alpha=1.0)  seed=0 ---
  fn before rescaling: 16.189
  fn after  rescaling: 16.189  (ratio: 1.00x)
  NC1 collapsed at epoch 320  fn=1.1084
  => T_NC=320  fn*=1.1084  (1551s)

--- condition=control (alpha=1.0)  seed=1 ---
  fn before rescaling: 16.627
  fn after  rescaling: 16.627  (ratio: 1.00x)
  NC1 collapsed at epoch 290  fn=1.0805
  => T_NC=290  fn*=1.0805  (1557s)

--- condition=control (alpha=1.0)  seed=2 ---
  fn before rescaling: 15.921
  fn after  rescaling: 15.921  (ratio: 1.00x)
  NC1 collapsed at epoch 250  fn=1.1202
  => T_NC=250  fn*=1.1202  (1547s)

--- condition=scale_down (alpha=0.3)  seed=0 ---
  fn before rescaling: 16.189
  fn after  rescaling: 4.857  (ratio: 0.30x)
  NC1 collapsed at epoch 320  fn=1.1308
  => T_NC=320  fn*=1.1308  (1547s)

--- condition=scale_down (alpha=0.3)  seed=1 ---
  fn before rescaling: 16.627
  fn after  rescaling: 4.988  (ratio: 0.30x)
  NC1 collapsed at epoch 320  fn=1.0814
  => T_

In [7]:
# ── Summary ────────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(all_results)
df_summary.to_csv(SAVE_DIR + 'intervention_summary.csv', index=False)

print('\n' + '='*55)
print('INTERVENTION EXPERIMENT SUMMARY')
print('='*55)
g = df_summary.groupby('condition').agg(
    alpha=('alpha', 'first'),
    T_NC_mean=('T_NC', 'mean'),
    T_NC_std=('T_NC', 'std'),
    fn_star_mean=('fn_at_TNC', 'mean'),
    fn_star_std=('fn_at_TNC', 'std'),
    n_collapsed=('T_NC', lambda x: x.notna().sum()),
).reset_index()

print(g.to_string(index=False))

print('\nInterpretation:')
ctrl = df_summary[df_summary['condition']=='control']['T_NC'].mean()
down = df_summary[df_summary['condition']=='scale_down']['T_NC'].mean()
up   = df_summary[df_summary['condition']=='scale_up']['T_NC'].mean()

if down < ctrl < up:
    print('  CAUSAL EVIDENCE: Scale DOWN accelerated collapse, Scale UP delayed it.')
    print(f'  Control T_NC={ctrl:.0f}  |  Scale DOWN T_NC={down:.0f}  |  Scale UP T_NC={up:.0f}')
elif abs(down - ctrl) < 20 and abs(up - ctrl) < 20:
    print('  CORRELATION EVIDENCE: All conditions collapsed at similar T_NC.')
    print('  fn* appears to be an equilibrium property, not a causal driver.')
else:
    print(f'  Mixed result: Control={ctrl:.0f}  Down={down:.0f}  Up={up:.0f}')
    print('  Requires further analysis.')



INTERVENTION EXPERIMENT SUMMARY
 condition  alpha  T_NC_mean  T_NC_std  fn_star_mean  fn_star_std  n_collapsed
   control    1.0 286.666667 35.118846      1.102994     0.020391            3
scale_down    0.3 316.666667  5.773503      1.093295     0.033186            3
  scale_up    3.0 316.666667 46.188022      1.054242     0.049295            3

Interpretation:
  Mixed result: Control=287  Down=317  Up=317
  Requires further analysis.
